In [1]:
import pickle

import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

import mlflow
import os

mlflow.set_tracking_uri(f"http://127.0.0.1:5000")

In [3]:
mlflow.search_model_versions(
    filter_string="name='nyc_taxi_trip_duration_model' and tags.stage='Production'"
)

[]

In [7]:
mlflow.sklearn.load_model("models:/nyc_taxi_trip_duration_model/1")

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['PULocationID',
                                                   'DOLocationID']),
                                                 ('num', StandardScaler(),
                                                  ['trip_distance'])])),
                ('regressor', LinearRegression())])

In [28]:
reg_model = mlflow.search_model_versions(filter_string="name='nyc_taxi_trip_duration_model' and tags.stage='Production'")[0]

In [32]:

logged_model = 'runs:/a894080f33a54b19a5393e8f7ab507e7/model'

# Load model
loaded_model = mlflow.pyfunc.load_model(logged_model)

MlflowException: Model does not have the "python_function" flavor

In [24]:
import mlflow.sklearn


model_name = "nyc_taxi_trip_duration_model"
model_run_id = reg_model.run_id

In [27]:

mlflow.sklearn.load_model(f'runs:/{model_run_id}/model')

TypeError: AssetDep.__new__() takes 2 positional arguments but 3 were given

In [3]:
mlflow.set_experiment("green-taxi-duration")

2025/06/01 23:48:15 INFO mlflow.tracking.fluent: Experiment with name 'green-taxi-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-artifacts-remote-100/2', creation_time=1748836095162, experiment_id='2', last_update_time=1748836095162, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [4]:

def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [6]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [7]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = mean_squared_error(y_pred, y_val, squared=False)
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

c:\Users\khanm375\Documents\mlops\venv\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725


2025/06/01 23:50:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run rogue-tern-908 at: http://ec2-3-99-156-154.ca-central-1.compute.amazonaws.com:5000/#/experiments/2/runs/69c892c1381b4810becddc0478f325c7
🧪 View experiment at: http://ec2-3-99-156-154.ca-central-1.compute.amazonaws.com:5000/#/experiments/2


In [8]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=f"http://{TRACKING_SERVER_HOST}:5000")

In [10]:
#list experiments
client.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-100/2', creation_time=1748836095162, experiment_id='2', last_update_time=1748836095162, lifecycle_stage='active', name='green-taxi-duration', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-100/1', creation_time=1747516860362, experiment_id='1', last_update_time=1747516860362, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-100/0', creation_time=1747514512791, experiment_id='0', last_update_time=1747514512791, lifecycle_stage='active', name='Default', tags={}>]

MlflowException: API request to endpoint /api/2.0/mlflow/registered-models/alias failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'